In [402]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.compose import ColumnTransformer

In [3]:
data = pd.read_csv('דאטה לפרויקט.csv')
data

,Age,Gender,Education,Person Income,Employee Experience,Home Onwership,Loan Amount,Loan Intent,Loan interest Rate,Loan percentage,Credit History,Credit Score,Previous Loan,Loan Status
0,22,female,Master,71948,0,RENT,35000,PERSONAL,16.02,0.49,3,561,No,1
1,21,female,High School,12282,0,OWN,1000,EDUCATION,11.14,0.08,2,504,Yes,0
2,25,female,High School,12438,3,MORTGAGE,5500,MEDICAL,12.87,0.44,3,635,No,1
3,23,female,Bachelor,79753,0,RENT,35000,MEDICAL,15.23,0.44,2,675,No,1
4,24,male,Master,66135,1,RENT,35000,MEDICAL,14.27,0.53,4,586,No,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44995,27,male,Associate,47971,6,RENT,15000,MEDICAL,15.66,0.31,3,645,No,1
44996,37,female,Associate,65800,17,RENT,9000,HOMEIMPROVEMENT,14.07,0.14,11,621,No,1
44997,33,male,Associate,56942,7,RENT,2771,DEBTCONSOLIDATION,10.02,0.05,10,668,No,1
44998,29,male,Bachelor,33164,4,RENT,12000,EDUCATION,13.23,0.36,6,604,No,1


In [300]:
X_1 = data[['Person Income', 'Home Onwership', 'Credit History', 'Credit Score', 'Loan percentage', 'Previous Loan']]
y = data['Loan Status'] 

X = pd.get_dummies(X_1, columns=['Home Onwership', 'Previous Loan'], drop_first=True)

numeric_features = ['Person Income', 'Credit History', 'Credit Score', 'Loan percentage']
categorical_features = ['Home Onwership_OTHER',	'Home Onwership_OWN', 'Home Onwership_RENT', 'Previous Loan_Yes']

scale_num_only = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features)
    ],
    remainder='passthrough' 
)

In [386]:
########### kernel='rbf'

loan_pipeline = Pipeline([
    ('scale_num_only', scale_num_only),
    ('svc', SVC(kernel='rbf'))
])

# Cross-validation
rbf_scores = cross_val_score(loan_pipeline, X, y, cv=3, scoring='accuracy', n_jobs=-1)

print("--- Cross-Validation Results ---")
print("Scores:", rbf_scores)
print(f"Average Accuracy: {rbf_scores.mean():.3f}")

# Indices calculation
y_cv_pred = cross_val_predict(loan_pipeline, X, y, cv=3, n_jobs=-1)
cm = confusion_matrix(y, y_cv_pred)
report = classification_report(y, y_cv_pred)

# Margin calculation :
pipe_rbf_trained = loan_pipeline.fit(X, y)
clf = pipe_rbf_trained.named_steps['svc']
sv_transformed = clf.support_vectors_
sv_scores = clf.decision_function(sv_transformed)
dual_coefs = clf.dual_coef_
w_norm = np.sqrt(np.dot(dual_coefs, sv_scores)[0])
np.dot(dual_coefs, sv_scores)[0]
geometric_margin = 2 / w_norm

print()
print('rbf kernel Indices:')
print("Confusion Matrix:\n",cm)
print(report)
print(f'The margin is: {geometric_margin}')

--- Cross-Validation Results ---
Scores: [0.8716     0.86666667 0.8752    ]
Average Accuracy: 0.871

rbf kernel Indices:
Confusion Matrix:
 [[33770  1230]
 [ 4568  5432]]
              precision    recall  f1-score   support

           0       0.88      0.96      0.92     35000
           1       0.82      0.54      0.65     10000

    accuracy                           0.87     45000
   macro avg       0.85      0.75      0.79     45000
weighted avg       0.87      0.87      0.86     45000

The margin is: 0.09353519423695916


In [400]:
joblib.dump(pipe_rbf_trained, 'loan_rbf_model.pkl')

['loan_rbf_model.pkl']

In [398]:
features_list = X_1.columns
print("The model's features are: \n", features_list)
print(f"Average Accuracy: {rbf_scores.mean():.3f}") 
print(f"Model file name: 'loan_rbf_model.pkl'")

The model's features are: 
 Index(['Person Income', 'Home Onwership', 'Credit History', 'Credit Score',
       'Loan percentage', 'Previous Loan'],
      dtype='object')
Average Accuracy: 0.871
Model file name: 'loan_rbf_model.pkl'


In [404]:
# Checking results

input_example = {
    'Person Income': 11000,
    'Credit History': 2,
    'Credit Score': 500,
    'Loan percentage': 0.12,
    'Home Onwership_OTHER': 0,
    'Home Onwership_OWN': 1,
    'Home Onwership_RENT': 0,
    'Previous Loan_Yes': 0
}
input_example_df = pd.DataFrame([input_example])
y_pred = pipe_rbf_trained.predict(input_example_df)
y_pred

array([1])

--- Rows to check , not for submission ---

In [339]:
########### kernel='linear'

loan_pipeline = Pipeline([
    ('scale_num_only', scale_num_only),
    ('svc', SVC(kernel='linear'))
])

# loan_pipeline = make_pipeline(
#     StandardScaler(),   
#     SVC(kernel='linear')    
# )

# Cross-validation
scores = cross_val_score(loan_pipeline, X, y, cv=3, scoring='accuracy')

print("--- Cross-Validation Results ---")
print("Scores:", scores)
print(f"Average Accuracy: {scores.mean():.3f}")

y_cv_pred = cross_val_predict(loan_pipeline, X, y, cv=3, n_jobs=-1)
cm = confusion_matrix(y, y_cv_pred)
report = classification_report(y, y_cv_pred)

# Margin calculation :
pipe_trained_linear = loan_pipeline.fit(X, y)
clf = pipe_trained_linear.named_steps['svc']
sv_transformed = clf.support_vectors_
sv_scores = clf.decision_function(sv_transformed)
dual_coefs = clf.dual_coef_
w_norm = np.sqrt(np.dot(dual_coefs, sv_scores)[0])
np.dot(dual_coefs, sv_scores)[0]
geometric_margin = 2 / w_norm

print()
print('linear kernel Indices:')
print("Confusion Matrix:\n",cm)
print(report)
print(f'The margin is: {geometric_margin}')

--- Cross-Validation Results ---
Scores: [0.86326667 0.8612     0.86833333]
Average Accuracy: 0.864

linear kernel Indices:
Confusion Matrix:
 [[32789  2211]
 [ 3897  6103]]
              precision    recall  f1-score   support

           0       0.89      0.94      0.91     35000
           1       0.73      0.61      0.67     10000

    accuracy                           0.86     45000
   macro avg       0.81      0.77      0.79     45000
weighted avg       0.86      0.86      0.86     45000

The margin is: 0.4678630926973437


In [340]:
trained_svc = pipe_trained_linear.named_steps['svc']
w = trained_svc.coef_[0]
b = trained_svc.intercept_[0]
display(w)
display(b)

array([-0.44823252,  0.01992754, -0.3813815 ,  0.69210619,  0.48296051,
       -0.85470475,  0.69569044, -4.        ])

np.float64(-0.7907052295167267)

In [149]:
########### kernel='poly' 

loan_pipeline = Pipeline([
    ('scale_num_only', scale_num_only),
    ('svc', SVC(kernel='poly'))
])

# loan_pipeline = make_pipeline(
#     StandardScaler(),   
#     SVC(kernel='poly')    
# )

# Cross-validation
scores = cross_val_score(loan_pipeline, X, y, cv=3, scoring='accuracy')

print("--- Cross-Validation Results ---")
print("Scores:", scores)
print(f"Average Accuracy: {scores.mean():.3f}")

y_cv_pred = cross_val_predict(loan_pipeline, X, y, cv=3, n_jobs=-1)
cm = confusion_matrix(y, y_cv_pred)
report = classification_report(y, y_cv_pred)

# Margin calculation :
pipe_poly_trained = loan_pipeline.fit(X, y)
clf = pipe_poly_trained.named_steps['svc']
sv_transformed = clf.support_vectors_
sv_scores = clf.decision_function(sv_transformed)
dual_coefs = clf.dual_coef_
w_norm = np.sqrt(np.dot(dual_coefs, sv_scores)[0])
np.dot(dual_coefs, sv_scores)[0]
geometric_margin = 2 / w_norm

print()
print('poly kernel Indices:')
print("Confusion Matrix:\n",cm)
print(report)
print(f'The margin is: {geometric_margin}')

--- Cross-Validation Results ---
Scores: [0.8508     0.84946667 0.86446667]
Average Accuracy: 0.855

poly kernel Indices:
Confusion Matrix:
 [[34024   976]
 [ 5553  4447]]
              precision    recall  f1-score   support

           0       0.86      0.97      0.91     35000
           1       0.82      0.44      0.58     10000

    accuracy                           0.85     45000
   macro avg       0.84      0.71      0.74     45000
weighted avg       0.85      0.85      0.84     45000

The margin is: 0.08195135023663008


In [32]:
########### kernel='sigmoid'

loan_pipeline = make_pipeline(
    StandardScaler(),   
    SVC(kernel='sigmoid')    
)

# Cross-validation
scores = cross_val_score(loan_pipeline, X, y, cv=3, scoring='accuracy')

print("--- Cross-Validation Results ---")
print("Scores:", scores)
print(f"Average Accuracy: {scores.mean():.3f}")

# pipe_trained = loan_pipeline.fit(X, y)  
# y_pred = pipe_trained.predict(X)
# cm = confusion_matrix(y, y_pred)
# report = classification_report(y, y_pred)

# # Margin calculation :
# clf = pipe_trained.named_steps['svc']
# sv_transformed = clf.support_vectors_
# sv_scores = clf.decision_function(sv_transformed)
# dual_coefs = clf.dual_coef_
# w_norm = np.sqrt(np.dot(dual_coefs, sv_scores)[0])
# np.dot(dual_coefs, sv_scores)[0]
# geometric_margin = 2 / w_norm

# print()
# print('sigmoid kernel Indices:')
# print("Confusion Matrix:\n",cm)
# print(report)
# print(f'The margin is: {geometric_margin}')

--- Cross-Validation Results ---
Scores: [0.71533333 0.75133333 0.7258    ]
Average Accuracy: 0.731


In [ ]:
# ####### kernel='poly'
# loan_pipeline_poly = make_pipeline(
#     StandardScaler(),   
#     SVC(kernel='poly')    
# )

# pipe_poly = loan_pipeline_poly.fit(X_train, y_train)  
# y_pred_poly = pipe_poly.predict(X_test)
# cm_poly = confusion_matrix(y_test, y_pred_poly)
# report_poly = classification_report(y_test, y_pred_poly)

# clf = pipe_poly.named_steps['svc']
# sv_transformed = clf.support_vectors_
# sv_scores = clf.decision_function(sv_transformed)
# dual_coefs = clf.dual_coef_

# w_norm = np.sqrt(np.dot(dual_coefs, sv_scores)[0])
# np.dot(dual_coefs, sv_scores)[0]
# geometric_margin_poly = 2 / w_norm

# print('poly kernel Indices:')
# print("Confusion Matrix:\n",cm_rbf)
# print(report_poly)
# print(f'The margin is: {geometric_margin_poly}')

# accuracy = accuracy_score(y_test, y_pred_poly)
# print('accuracy: ', accuracy)

In [ ]:
# 1. שליפת המודל הפולינומי מתוך ה-Pipeline
poly_svc = loan_pipeline_poly.named_steps['svc']

# 2. שליפת הווקטורים התומכים שכבר עברו טרנספורמציה ונרמול
sv_transformed = poly_svc.support_vectors_

# 3. חישוב פונקציית ההחלטה הלא-מנורמלת רק עבור הווקטורים התומכים
# (מכיוון שהם כבר מנורמלים, אנו קוראים למודל הפנימי ישירות כדי לעקוף את ה-Pipeline)
sv_scores = poly_svc.decision_function(sv_transformed)

# 4. חילוץ המקדמים הדואליים (alpha * y)
dual_coefs = poly_svc.dual_coef_

# 5. חישוב מהיר של אורך ווקטור המשקלים הווירטואלי (||w||) במרחב הגבוה
w_norm = np.sqrt(np.dot(dual_coefs, sv_scores)[0])

# 6. חישוב המרווח (Margin) הגיאומטרי
geometric_margin_poly = 2 / w_norm

print(f"גודל ה-Margin במרחב הפולינומי: {geometric_margin_poly:.4f}")


In [296]:
data[(data['Loan Status'] == 1) ][['Person Income', 'Home Onwership', 'Credit History', 'Credit Score', 'Previous Loan', 'Loan percentage', 'Loan interest Rate']].sort_values('Person Income')

,Person Income,Home Onwership,Credit History,Credit Score,Previous Loan,Loan percentage,Loan interest Rate
44274,8000,RENT,13,696,No,0.15,12.43
43881,8000,RENT,16,640,No,0.16,12.16
15960,8000,RENT,2,643,No,0.22,14.84
15961,8000,RENT,2,687,No,0.15,14.26
15962,8000,RENT,4,597,No,0.30,16.00
...,...,...,...,...,...,...,...
44008,725801,MORTGAGE,9,733,No,0.03,9.01
44922,726416,MORTGAGE,9,634,No,0.03,9.07
43915,736127,MORTGAGE,9,623,No,0.03,10.60
17840,778515,MORTGAGE,8,705,No,0.01,17.19


In [297]:
data[(data['Loan Status'] == 0) & (data['Previous Loan']== 'Yes') ][['Person Income', 'Home Onwership', 'Credit History', 'Credit Score', 'Previous Loan', 'Loan Amount', 'Loan interest Rate']].sort_values('Person Income')

,Person Income,Home Onwership,Credit History,Credit Score,Previous Loan,Loan Amount,Loan interest Rate
35288,9595,RENT,12,620,Yes,912,11.06
31928,9850,RENT,13,473,Yes,1000,11.14
15951,10161,MORTGAGE,3,544,Yes,500,10.71
35075,10206,RENT,3,623,Yes,1196,15.10
16821,10482,RENT,4,606,Yes,1000,11.36
...,...,...,...,...,...,...,...
35850,1661567,MORTGAGE,17,622,Yes,6545,7.65
31924,1728974,MORTGAGE,15,573,Yes,6400,7.40
41288,1741243,MORTGAGE,18,655,Yes,12011,10.42
32546,2280980,MORTGAGE,21,682,Yes,1500,11.01


In [212]:
data[ (data['Previous Loan']== 'No')][['Person Income', 'Home Onwership', 'Credit History', 'Credit Score', 'Previous Loan', 'Loan Amount', 'Loan interest Rate']]

,Person Income,Home Onwership,Credit History,Credit Score,Previous Loan,Loan Amount,Loan interest Rate
0,71948,RENT,3,561,No,35000,16.02
2,12438,MORTGAGE,3,635,No,5500,12.87
3,79753,RENT,2,675,No,35000,15.23
4,66135,RENT,4,586,No,35000,14.27
5,12951,OWN,2,532,No,2500,7.14
...,...,...,...,...,...,...,...
44995,47971,RENT,3,645,No,15000,15.66
44996,65800,RENT,11,621,No,9000,14.07
44997,56942,RENT,10,668,No,2771,10.02
44998,33164,RENT,6,604,No,12000,13.23


In [ ]:
([-2.00350256,  0.00561976, -0.33594468,  0.60814744,  0.04824866,
       -0.19211933,  0.46193483, -1.63817987])

In [282]:
# trained_svc = loan_pipeline_linear.named_steps['svc']
# w = trained_svc.coef_[0]
# b = trained_svc.intercept_[0]
# display(w)
# display(b)

array([-2.00350256,  0.00561976, -0.33594468,  0.60814744,  0.04824866,
       -0.19211933,  0.46193483, -1.63817987])

np.float64(-2.231017374448146)